# Heterogeneous Treatment Effects: CATE with `econml`

**Goal:** Estimate CATE using `CausalForestDML`, run GATES analysis, and visualise treatment effect heterogeneity across commodity sectors.

**Time:** 15 minutes

You will estimate individual-level treatment effects for inventory surprises across energy, metals, and agriculture futures.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from econml.dml import CausalForestDML
from sklearn.ensemble import GradientBoostingRegressor

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Simulate: Inventory Surprises Across Commodity Sectors

The treatment effect varies by sector: energy is highly sensitive, metals moderate, agriculture negligible.

In [ ]:
n = 5000
p = 20

X = np.random.randn(n, p)
sector = np.random.choice([0, 1, 2], n, p=[0.4, 0.3, 0.3])
sector_names = {0: 'Energy', 1: 'Metals', 2: 'Agriculture'}

# Additional controls including sector dummies
W = np.column_stack([X, (sector == 0).astype(float),
                     (sector == 1).astype(float)])

# Treatment: inventory surprise
D = 0.5 * X[:, 0] + 0.3 * X[:, 1] + np.random.randn(n) * 0.5

# Heterogeneous CATE
tau = np.where(sector == 0, 1.2 + 0.3 * X[:, 0],
       np.where(sector == 1, 0.3 + 0.1 * X[:, 0],
                -0.1 + 0.05 * X[:, 0]))

Y = tau * D + 0.5 * X[:, 0] + 0.3 * X[:, 2] + np.random.randn(n) * 0.5

print('True CATE by sector:')
for s in range(3):
    print(f'  {sector_names[s]:<15} mean={np.mean(tau[sector == s]):.3f}')
print(f'  Overall ATE:    {np.mean(tau):.3f}')

## Estimate CATE with CausalForestDML

Use a causal forest with gradient boosting for the nuisance functions.

In [ ]:
cf_dml = CausalForestDML(
    model_y=GradientBoostingRegressor(200, max_depth=5, random_state=42),
    model_t=GradientBoostingRegressor(200, max_depth=5, random_state=42),
    n_estimators=200,
    random_state=42
)
cf_dml.fit(Y, D, X=X, W=W)

cate_hat = cf_dml.effect(X)

print('Estimated CATE by sector:')
for s in range(3):
    mask = sector == s
    print(f'  {sector_names[s]:<15} true={np.mean(tau[mask]):.3f}  '
          f'est={np.mean(cate_hat[mask]):.3f}')
print(f'  Overall ATE:    true={np.mean(tau):.3f}  est={np.mean(cate_hat):.3f}')

## GATES Analysis: Sorted Group Effects

Sort observations by estimated CATE, divide into quintiles, and compute group averages.

In [ ]:
# GATES: sort by estimated CATE
sorted_idx = np.argsort(cate_hat)
n_groups = 5
group_size = n // n_groups

gates = []
for g in range(n_groups):
    start = g * group_size
    end = (g + 1) * group_size if g < n_groups - 1 else n
    idx = sorted_idx[start:end]
    gates.append({
        'Quintile': g + 1,
        'Est_CATE': np.mean(cate_hat[idx]),
        'True_CATE': np.mean(tau[idx]),
        'N': len(idx),
        'Pct_Energy': np.mean(sector[idx] == 0),
    })

gates_df = pd.DataFrame(gates)
print(gates_df.to_string(index=False))
print('\nMonotone pattern confirms real heterogeneity.')
print('Top quintile is dominated by energy sector observations.')

## Visualise: CATE Distribution and GATES

Plot the CATE distribution by sector and the GATES bar chart.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: CATE distribution by sector
for s in range(3):
    mask = sector == s
    axes[0].hist(cate_hat[mask], bins=30, alpha=0.5,
                 label=f'{sector_names[s]} (true={np.mean(tau[mask]):.2f})',
                 density=True)
axes[0].set_xlabel('Estimated CATE', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('CATE Distribution by Sector', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Right: GATES
x_pos = gates_df['Quintile']
w = 0.3
axes[1].bar(x_pos - w/2, gates_df['True_CATE'], w,
            label='True', color='crimson', alpha=0.7)
axes[1].bar(x_pos + w/2, gates_df['Est_CATE'], w,
            label='Estimated', color='steelblue', alpha=0.7)
axes[1].set_xlabel('CATE Quintile', fontsize=12)
axes[1].set_ylabel('Group Average Treatment Effect', fontsize=12)
axes[1].set_title('GATES: Sorted Group Effects', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Summary

**What you learned:**
1. CATE reveals treatment effect heterogeneity -- who benefits most
2. `CausalForestDML` combines DML residualisation with causal forests
3. GATES analysis provides a simple test for monotone heterogeneity
4. Energy futures are 4x more sensitive to inventory surprises than metals

**What is next:**
- **Module 09:** Production DML pipeline with automated diagnostics

**Key takeaway:** CATE is the most actionable output for commodity traders.
It tells you not just IF a treatment works, but FOR WHOM it works best.

In [ ]:
learning_objectives(["CATE reveals treatment effect heterogeneity -- who benefits most", "`CausalForestDML` combines DML residualisation with causal forests", "GATES analysis provides a simple test for monotone heterogeneity", "Energy futures are 4x more sensitive to inventory surprises than metals"])

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])